# Exploratory Data Analysis (EDA)

In this notebook, we will explore the Appliance Energy Prediction Dataset. The main objectives are:
1. Data Loading & Understanding
2. Summary Statistics
3. Handling Missing Values
4. Distribution Analysis
5. Correlation Matrix & Time-Series Trends

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Set plotting style
sns.set_style('whitegrid')
plt.rcParams['figure.figsize'] = (12, 6)

In [ ]:
# Load the dataset
data_path = '../data/raw/energy_data_set.csv'
try:
    df = pd.read_csv(data_path)
    # Convert simply 'date' column to datetime
    if 'date' in df.columns:
        df['date'] = pd.to_datetime(df['date'])
        df.set_index('date', inplace=True)
    
    print(f"Dataset shape: {df.shape}")
    display(df.head())
except FileNotFoundError:
    print(f"File not found at {data_path}. Please ensure the dataset is downloaded.")

### 1. Data Understanding & Summary Statistics
Let's check the data types and basic statistics to understand the scale of our features.

In [ ]:
# Check data types and for any missing values directly
print(df.info())

# Basic summary statistics
display(df.describe())

### 2. Handling Missing Values
Checking the exact count of missing values per column. In time-series, missing values are usually handled via interpolation or forward/backward filling rather than dropping rows, to maintain temporal sequence.

In [ ]:
missing_values = df.isnull().sum()
print("Missing values per column:\n", missing_values[missing_values > 0])

# Just in case, let's set up an interpolation method for any missing time-series points
if missing_values.sum() > 0:
    print("Interpolating missing values...")
    df = df.interpolate(method='time')
else:
    print("No missing values found.")

### 3. Distribution Analysis
Visualizing the distribution of our target variable `Appliances` (energy consumption) and `Lights`.

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(16, 5))

# Target Variable (Appliances)
sns.histplot(df['Appliances'], bins=50, kde=True, ax=ax[0], color='blue')
ax[0].set_title('Distribution of Appliance Energy Consumption (Wh)')

# Lights Variable
if 'lights' in df.columns:
    sns.histplot(df['lights'], bins=50, kde=True, ax=ax[1], color='orange')
    ax[1].set_title('Distribution of Lights Energy Consumption (Wh)')
else:
    print("Column 'lights' not found, skipping its plot.")

plt.show()

### 4. Correlation Matrix
Checking for multicollinearity among the temperature and humidity sensors, and how they correlate with the target variable `Appliances`.

In [ ]:
plt.figure(figsize=(16, 12))
# Only correlate numeric columns
corr_matrix = df.select_dtypes(include=[np.number]).corr()

# Mask the upper triangle for easier readability
mask = np.triu(np.ones_like(corr_matrix, dtype=bool))
sns.heatmap(corr_matrix, mask=mask, annot=False, cmap='coolwarm', linewidths=0.5)
plt.title('Correlation Matrix of All Features')
plt.show()

# Let's see the top variables correlated with Appliances
print("Top correlations with Appliances:")
print(corr_matrix['Appliances'].sort_values(ascending=False).head(10))
print("\nBottom correlations with Appliances:")
print(corr_matrix['Appliances'].sort_values(ascending=True).head(10))